In [1]:
from urllib import response

import pandas as pd
from packaging.utils import parse_wheel_filename

In [2]:
vessels = pd.read_csv('../data/shadow_fleet_vessels.csv')
print(vessels.shape)
vessels.head()

(50, 9)


,imo,vessel_name,tanker_size,build_year,flag,ship_manager,registered_owner,ism_manager,list_source
0,9328170,Aether,Aframax,2007,Unknown,UAE. Zulu Ships Management,UAE. Wrasse Ship Co,UAE. Zulu Ships Management,new_crude_2025
1,9282481,Tagor,Aframax,2005,Unknown,UAE. Zulu Ships Management,UAE. Blaire Charters Ltd,UAE. Tarabya Logistics Ltd,new_crude_2025
2,9236248,Phoenix I,Aframax,2002,Comoros,UAE. Triviality Shipping Llc,UAE. Celestial Overseas Co Ltd,UAE. Triviality Shipping Llc,new_crude_2025
3,9439383,Bhilva,Aframax,2010,Panama,Seychelles. Tika Shipping Ltd,Seychelles. Tika Shipping Ltd,China. Song Hua Jiang Shipmanagement,new_crude_2025
4,9311610,Torx,Suezmax,2006,Panama,Seychelles. Shiva Shipping Ltd,Seychelles. Shiva Shipping Ltd,Singapore. Crest Maritime Pte Ltd,new_crude_2025


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.getenv("GFW_API_TOKEN")
print("token loaded:", token is not None and len(token) > 20)

token loaded: True


In [4]:
import requests
headers = {
    "Authorization": f"Bearer {token}"
}
imo_test = "9328170"
url = f"https://gateway.api.globalfishingwatch.org/v3/vessels/search"
params = {
    "query": imo_test,
    "datasets[0]": "public-global-vessel-identity:latest"
}

response = requests.get(url, headers=headers, params=params)
print("status", response.status_code)
data = response.json()
data

status 200


{'limit': 30,
 'since': None,
 'total': 7,
 'entries': [{'dataset': 'public-global-vessel-identity:v4.0',
   'registryInfoTotalRecords': 1,
   'registryInfo': [{'id': '3f4aa69a84a907927d2d005eedfc209a',
     'sourceCode': ['TMT_OTHER'],
     'ssvid': '352002230',
     'flag': 'PAN',
     'shipname': 'AETHER',
     'nShipname': 'AETHER',
     'callsign': '3E2221',
     'imo': '9328170',
     'latestVesselInfo': True,
     'transmissionDateFrom': '2023-01-23T14:38:25Z',
     'transmissionDateTo': '2025-10-06T11:09:01Z',
     'lengthM': None,
     'tonnageGt': None,
     'vesselInfoReference': 'e974a02026523f16c8fb65f7f2418021',
     'extraFields': [{'registrySource': 'TMT',
       'iuuStatus': {'value': None,
        'dateFrom': None,
        'dateFromMask': None,
        'dateTo': None,
        'dateToMask': None},
       'hasComplianceInfo': None,
       'images': [],
       'operator': None,
       'builtYear': {'value': None,
        'dateFrom': None,
        'dateFromMask': None,
  

In [5]:
def get_vessel_info(imo):
    url = "https://gateway.api.globalfishingwatch.org/v3/vessels/search"
    params = {
        "query": imo,
        "datasets[0]": "public-global-vessel-identity:latest"
    }
    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        print(f"Error for imo {imo}: status code {response.status_code}")
        return None
    return response.json()

In [6]:
result = get_vessel_info("9282481")
result

{'limit': 30,
 'since': None,
 'total': 8,
 'entries': [{'dataset': 'public-global-vessel-identity:v4.0',
   'registryInfoTotalRecords': 1,
   'registryInfo': [{'id': 'b7fd24234754a83983fe30f270904aca',
     'sourceCode': ['TMT_OTHER'],
     'ssvid': '352002249',
     'flag': 'PAN',
     'shipname': 'TAGOR',
     'nShipname': 'TAGOR',
     'callsign': '3E2235',
     'imo': '9282481',
     'latestVesselInfo': True,
     'transmissionDateFrom': '2024-07-06T08:23:17Z',
     'transmissionDateTo': '2025-09-12T05:12:57Z',
     'lengthM': None,
     'tonnageGt': None,
     'vesselInfoReference': '5b0b2edf826c8b7f49b49c48cef20593',
     'extraFields': [{'registrySource': 'TMT',
       'iuuStatus': {'value': None,
        'dateFrom': None,
        'dateFromMask': None,
        'dateTo': None,
        'dateToMask': None},
       'hasComplianceInfo': None,
       'images': [],
       'operator': None,
       'builtYear': {'value': None,
        'dateFrom': None,
        'dateFromMask': None,
    

In [15]:
def parse_vessel_data(imo, raw_data):
    if raw_data is None or raw_data['total'] == 0:
        return {"imo": imo, "found": False}, []

    all_identities = []

    for entry in raw_data['entries']:

        source_records = entry['registryInfo'] if entry['registryInfo'] else entry['selfReportedInfo']

        for record in source_records:
            all_identities.append({
                "imo": imo,
                "shipname": record.get('shipname'),
                "flag": record.get('flag'),
                "ssvid": record.get('ssvid'),
                "match_fields": record.get('matchFields', 'REGISTRY'),
                "transmission_from": record.get('transmissionDateFrom'),
                "transmission_to": record.get('transmissionDateTo'),
                "is_verified": record.get('matchFields') == 'SEVERAL_FIELDS' or 'sourceCode' in record and record.get('sourceCode') == ['IMO'],
            })

    # summary only verificated
    verified = [r for r in all_identities if r['is_verified']]
    current = max(verified, key=lambda r: r['transmission_to'] or '') if verified else all_identities[0]

    summary = {
        "imo": imo,
        "found": True,
        "current_name": current['shipname'],
        "current_flag": current['flag'],
        "num_total_identities": len(all_identities),
        "num_verified_identities": len(verified),
        "num_suspicious_matches": len(all_identities) - len(verified),
    }

    return summary, all_identities

In [18]:
summary, history = parse_vessel_data("9282481", result)
print(summary)

{'imo': '9282481', 'found': True, 'current_name': 'BRITISHGANNET', 'current_flag': 'IMN', 'num_total_identities': 8, 'num_verified_identities': 1, 'num_suspicious_matches': 7}


In [22]:
all_summaries = []
all_identities_full = []

for imo in vessels['imo']:
    raw = get_vessel_info(imo)
    summary, indentities = parse_vessel_data(imo, raw)
    all_summaries.append(summary)
    all_identities_full.extend(indentities)

In [24]:
for a in all_summaries:
    print(a)

{'imo': 9328170, 'found': True, 'current_name': 'WAFRAH', 'current_flag': 'KWT', 'num_total_identities': 7, 'num_verified_identities': 1, 'num_suspicious_matches': 6}
{'imo': 9282481, 'found': True, 'current_name': 'BRITISHGANNET', 'current_flag': 'IMN', 'num_total_identities': 8, 'num_verified_identities': 1, 'num_suspicious_matches': 7}
{'imo': 9236248, 'found': True, 'current_name': 'MINERVAZENIA', 'current_flag': 'GRC', 'num_total_identities': 7, 'num_verified_identities': 1, 'num_suspicious_matches': 6}
{'imo': 9439383, 'found': True, 'current_name': 'GIOVANNIBATTISTADECARLINI', 'current_flag': 'ITA', 'num_total_identities': 4, 'num_verified_identities': 1, 'num_suspicious_matches': 3}
{'imo': 9311610, 'found': True, 'current_name': 'MILTIADISM2', 'current_flag': 'LBR', 'num_total_identities': 3, 'num_verified_identities': 1, 'num_suspicious_matches': 2}
{'imo': 9472634, 'found': True, 'current_name': 'ADVANTAGEANTHEM', 'current_flag': 'MHL', 'num_total_identities': 2, 'num_verifi

In [25]:
for b in all_identities_full:
    print(b)

{'imo': 9328170, 'shipname': 'AETHER', 'flag': 'PAN', 'ssvid': '352002230', 'match_fields': 'REGISTRY', 'transmission_from': '2023-01-23T14:38:25Z', 'transmission_to': '2025-10-06T11:09:01Z', 'is_verified': False}
{'imo': 9328170, 'shipname': 'WAFRAH', 'flag': 'KWT', 'ssvid': '447162000', 'match_fields': 'REGISTRY', 'transmission_from': '2012-01-08T14:12:44Z', 'transmission_to': '2022-05-10T11:17:25Z', 'is_verified': True}
{'imo': 9328170, 'shipname': 'AETHER', 'flag': 'MDG', 'ssvid': '647553007', 'match_fields': 'NO_MATCH', 'transmission_from': '2026-03-14T15:43:17Z', 'transmission_to': '2026-05-19T14:51:02Z', 'is_verified': False}
{'imo': 9328170, 'shipname': 'AETHER', 'flag': 'GIN', 'ssvid': '632001155', 'match_fields': 'NO_MATCH', 'transmission_from': '2025-10-06T11:17:25Z', 'transmission_to': '2026-03-14T15:49:37Z', 'is_verified': False}
{'imo': 9328170, 'shipname': 'AETHER', 'flag': 'CMR', 'ssvid': '613442802', 'match_fields': 'NO_MATCH', 'transmission_from': '2026-05-19T14:56:47

In [26]:
summary_df = pd.DataFrame(all_summaries)
identities_df = pd.DataFrame(all_identities_full)

print(summary_df.shape)
print(identities_df.shape)
summary_df.head()

(50, 7)
(163, 8)


,imo,found,current_name,current_flag,num_total_identities,num_verified_identities,num_suspicious_matches
0,9328170,True,WAFRAH,KWT,7,1,6
1,9282481,True,BRITISHGANNET,IMN,8,1,7
2,9236248,True,MINERVAZENIA,GRC,7,1,6
3,9439383,True,GIOVANNIBATTISTADECARLINI,ITA,4,1,3
4,9311610,True,MILTIADISM2,LBR,3,1,2


In [27]:
identities_df['flag'].value_counts()

flag
PAN    24
MHL    20
RUS    17
LBR    16
CMR    11
MLT     6
SLE     6
GRC     5
BRB     5
MDG     4
HKG     4
GIN     3
COM     3
BHS     3
PLW     3
SGP     3
NIC     2
GNQ     2
DEU     2
AZE     2
CHN     2
TUR     2
KWT     1
IMN     1
FRA     1
COK     1
MOZ     1
BES     1
ITA     1
VUT     1
SMR     1
DNK     1
ESP     1
CYM     1
ATG     1
CYP     1
SAU     1
SYR     1
Name: count, dtype: int64

In [28]:
summary_df['num_suspicious_matches'].describe()

count    50.00000
mean      2.54000
std       1.82063
min       0.00000
25%       1.00000
50%       2.00000
75%       3.75000
max       7.00000
Name: num_suspicious_matches, dtype: float64